# Week 3 Lab — Regression Analysis
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


## 1. Learning Objectives

By the end of this lab, you should be able to:

- Fit and evaluate an ordinary least squares (OLS) regression model, and a multiple-feature linear model.
- Diagnose underfitting and overfitting using polynomial regression at different degrees.
- Apply Ridge and Lasso regularization and explain their effect on model coefficients and test error.
- Train a random forest regressor and compare it against linear approaches.
- Choose and justify a regressor for a sensor-calibration task, backed by RMSE/R² numbers.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("Environment OK")

## 3. Part A — Force Sensor Calibration Data

A strain-gauge outputs a raw voltage; the true relationship to applied force is mildly nonlinear (as is common with real sensors near the edge of their range), plus measurement noise.

In [ ]:
def generate_calibration_data(n_samples=120, seed=0):
    rng = np.random.default_rng(seed)
    voltage = np.sort(rng.uniform(0, 5, n_samples))
    true_force = 2.5 * voltage + 0.8 * voltage**2
    noise = rng.normal(0, 1.2, n_samples)
    force = true_force + noise
    return pd.DataFrame({"voltage": voltage, "force": force})


cal_df = generate_calibration_data()
plt.figure(figsize=(6, 4))
plt.scatter(cal_df["voltage"], cal_df["force"], s=15, color="#3E5C76")
plt.xlabel("Voltage (V)")
plt.ylabel("Force (N)")
plt.title("Raw calibration data")
plt.tight_layout()
plt.show()

## 4. Part B — A First (Underfitting) Baseline: Plain OLS

Fit a plain linear regression, `force ~ voltage`, and look at both the fit and the residuals.

In [ ]:
X = cal_df[["voltage"]].values
y = cal_df["force"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# TODO: fill in the linear regression 
lin_reg = None
y_pred = lin_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"Linear (degree 1)  RMSE: {rmse:.3f}   R^2: {r2:.3f}")

voltage_range = np.linspace(0, 5, 200).reshape(-1, 1)
plt.figure(figsize=(6, 4))
plt.scatter(cal_df["voltage"], cal_df["force"], s=15, color="#3E5C76", label="data")
plt.plot(voltage_range, lin_reg.predict(voltage_range), color="#FF6A39", linewidth=2, label="linear fit")
plt.xlabel("Voltage (V)"); plt.ylabel("Force (N)"); plt.legend()
plt.title("Plain linear regression")
plt.tight_layout()
plt.show()

Look at the plot: does a straight line capture the curvature in the data? Keep your RMSE here — you'll compare every later model against it.

## 5. Part C — Multiple Linear Regression: Friction Torque

A second, independent dataset: predict joint friction torque from velocity, temperature, and load. This relationship is linear by construction, so a plain linear model should do well — and its coefficients should roughly recover the numbers used to generate the data.

In [ ]:
def generate_friction_data(n_samples=200, seed=1):
    rng = np.random.default_rng(seed)
    velocity = rng.uniform(0, 10, n_samples)
    temperature = rng.uniform(20, 80, n_samples)
    load = rng.uniform(0, 50, n_samples)
    friction_torque = 3.0 + 0.6 * velocity + 0.15 * temperature - 0.05 * load 
    friction_torque += rng.normal(0, 0.8, n_samples) 
    return pd.DataFrame({
        "velocity": velocity, "temperature": temperature, "load": load,
        "friction_torque": friction_torque,
    })


friction_df = generate_friction_data()

Xf = friction_df[["velocity", "temperature", "load"]].values
yf = friction_df["friction_torque"].values
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.3, random_state=0)

friction_model = LinearRegression().fit(Xf_train, yf_train)
yf_pred = friction_model.predict(Xf_test)

print(f"RMSE: {np.sqrt(mean_squared_error(yf_test, yf_pred)):.3f}   R^2: {r2_score(yf_test, yf_pred):.3f}")
print()
for name, coef in zip(["velocity", "temperature", "load"], friction_model.coef_):
    print(f"{name:12s}: {coef:+.3f}")
print(f"{'intercept':12s}: {friction_model.intercept_:+.3f}")
print()
print("(Generated with true coefficients: velocity=+0.6, temperature=+0.15, load=-0.05, intercept=3.0)")

## 6. Part D — Polynomial Regression & the Overfitting Problem

Back to the calibration data. Fit polynomial regression at several degrees, using `PolynomialFeatures` + `LinearRegression` in a `Pipeline` (same pattern as Week 2's `ColumnTransformer` — chain preprocessing and the model together).

Fill in the `TODO` below: for each degree, fit the pipeline and record **both** the training RMSE and the test RMSE.

In [ ]:
# You can compute RMSE as in the previous code, or use the following function 
from sklearn.metrics import root_mean_squared_error

In [ ]:
degrees = [1, 2, 3, 5, 10, 12, 14, 16]
train_rmse, test_rmse = [], []

for deg in degrees:
    poly_pipe = Pipeline(steps=[
        ("poly", PolynomialFeatures(degree=deg)),
        ("linreg", LinearRegression()),
    ])
    poly_pipe.fit(X_train, y_train)

    # TODO: compute RMSE on the TRAINING set and on the TEST set for this pipeline.
    # Hint: poly_pipe.predict(X_train) and poly_pipe.predict(X_test)
    deg_train_rmse = None
    deg_test_rmse = None

    train_rmse.append(deg_train_rmse)
    test_rmse.append(deg_test_rmse)
    print(f"degree {deg:2d}   train RMSE: {deg_train_rmse}   test RMSE: {deg_test_rmse}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(degrees, train_rmse, marker="o", color="#3E5C76", label="train RMSE")
plt.plot(degrees, test_rmse, marker="o", color="#FF6A39", label="test RMSE")
plt.xlabel("Polynomial degree"); plt.ylabel("RMSE"); plt.legend()
plt.title("Train vs. test error as model flexibility increases")
plt.tight_layout()
plt.show()

> **What to look for:** training error should keep dropping (or flatten) as degree increases — the model can always fit the training points better with more flexibility. Test error should drop, then start rising again. The degree where they diverge is where overfitting begins.

## 7. Part E — Taming Overfitting with Ridge & Lasso

Take the overfit **degree-12**  polynomial and apply Ridge and Lasso instead of plain linear regression. Because regularization penalizes coefficient size directly, **scale the polynomial features first** — same rule as Week 2.

Fill in the `TODO` below.

In [ ]:
# TODO: build two Pipelines, ridge_pipe and lasso_pipe, each with three steps:
#   1. PolynomialFeatures(degree=12)
#   2. StandardScaler()
#   3. Ridge(alpha=0.1)   /   Lasso(alpha=0.05)
# Fit both on X_train, y_train.

ridge_pipe = None
lasso_pipe = None

ridge_pipe.fit(X_train, y_train)
lasso_pipe.fit(X_train, y_train)

for name, pipe in [("Ridge (degree 12)", ridge_pipe), ("Lasso (degree 12)", lasso_pipe)]:
    y_pred = pipe.predict(X_test)
    print(f"{name:20s} test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}"
          f"   R^2: {r2_score(y_test, y_pred):.3f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(cal_df["voltage"], cal_df["force"], s=15, color="#3E5C76", label="data", zorder=3)

unregularized_pipe = Pipeline(steps=[("poly", PolynomialFeatures(degree=12)), ("linreg", LinearRegression())])
unregularized_pipe.fit(X_train, y_train)

for name, pipe, color in [
    ("degree 12, no regularization", unregularized_pipe, "#6B4F9C"),
    ("degree 12, Ridge", ridge_pipe, "#FF6A39"),
    ("degree 12, Lasso", lasso_pipe, "#1B2A41"),
]:
    plt.plot(voltage_range.ravel(), pipe.predict(voltage_range), color=color, linewidth=2, label=name)

plt.xlabel("Voltage (V)"); plt.ylabel("Force (N)"); plt.legend()
plt.title("Regularization taming a degree-12 polynomial")
plt.ylim(cal_df["force"].min() - 3, cal_df["force"].max() + 3)
plt.tight_layout()
plt.show()

## 8. Part F — Random Forest Regression

Fit a random forest on the raw (unscaled, non-polynomial) voltage feature and compare.

In [ ]:
# TODO :create random forest regressor with desired hyperparameters, for example, 200 estimators, maximum depth of 5
rf = None
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(f"Random Forest   test RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.3f}"
      f"   R^2: {r2_score(y_test, rf_pred):.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(cal_df["voltage"], cal_df["force"], s=15, color="#3E5C76", label="data")
plt.plot(voltage_range.ravel(), rf.predict(voltage_range), color="#FF6A39", linewidth=2, label="random forest")
plt.xlabel("Voltage (V)"); plt.ylabel("Force (N)"); plt.legend()
plt.title("Random forest fit")
plt.tight_layout()
plt.show()

## 9. Part G — Final Comparison

Collect the test RMSE for every model you've built this lab and compare them side by side.

In [ ]:
final_results = {
    "Linear (deg 1)": np.sqrt(mean_squared_error(y_test, lin_reg.predict(X_test))),
    "Poly deg 12, none": np.sqrt(mean_squared_error(y_test, unregularized_pipe.predict(X_test))),
    "Poly deg 12, Ridge": np.sqrt(mean_squared_error(y_test, ridge_pipe.predict(X_test))),
    "Poly deg 12, Lasso": np.sqrt(mean_squared_error(y_test, lasso_pipe.predict(X_test))),
    "Random Forest": np.sqrt(mean_squared_error(y_test, rf.predict(X_test))),
}

plt.figure(figsize=(7, 4))
plt.bar(final_results.keys(), final_results.values(), color=["#3E5C76", "#6B4F9C", "#FF6A39", "#1B2A41", "#8A97A8"])
plt.ylabel("Test RMSE (N)")
plt.title("Final model comparison — force sensor calibration")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

for name, rmse_val in sorted(final_results.items(), key=lambda kv: kv[1]):
    print(f"{name:20s} RMSE: {rmse_val:.3f}")

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. Why did the plain linear model (degree 1) underfit the calibration curve? What in the RMSE numbers and the plot tells you that?**

*Your answer:* 

**2. What happened to the degree-12 model's train vs. test RMSE as degree increased, and why is a growing gap between them the signature of overfitting?**

*Your answer:* 

**3. How did Ridge and Lasso change the degree-12 model's test performance compared to the unregularized version? Which would you pick, and why?**

*Your answer:* 

**4. The random forest didn't need `PolynomialFeatures` or scaling to fit the curve well. Why not — and what's one situation where you'd still prefer a polynomial or linear model over it, for this sensor-calibration task?**

*Your answer:*  

## 11. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The train-vs-test RMSE plot from §6, and the regularization comparison plot from §7.
- The final comparison bar chart from §9.
- Your written answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| OLS baseline and multiple linear regression correctly fit & evaluated | 20% |
| Polynomial degree sweep correctly implemented and interpreted | 25% |
| Ridge / Lasso pipelines correctly built and compared | 25% |
| Random forest comparison and final write-up | 20% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

---
**Next week:** *Ensemble Learning* — bagging, random forests, gradient boosting, and XGBoost, applied to predictive maintenance.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>
